# Introduction to parallel arrays

ParallelArray is a high-level interface for distributed management of chunked arrays built on top of Zarr arrays. 
It is designed to operate seamlessly in Message Passing Interface (MPI) environments, enabling efficient 
parallel read, write, and mathematical operations distributed across multiple MPI processes.

The class maps array chunks to MPI ranks to balance memory and workload distribution, supports chunk-wise data access 
and modification with automatic MPI synchronization, and ensures data consistency through collective MPI 
communication. This enables scalable and robust handling of large datasets in high-performance computing workflows.

If you're unfamiliar with Zarr arrays, we recommend exploring the
[Zarr user guide for arrays](https://zarr.readthedocs.io/en/stable/user-guide/arrays.html)
to gain a deeper understanding of its fundamentals.

Let's explore the ParallelArray class by creating a small interactive MPI (Message Passing Interface)
cluster in this notebook using [ipyparallel](https://ipyparallel.readthedocs.io/). We'll create 4 parallel processes:

In [1]:
import ipyparallel as ipp

# Create an MPI cluster with 4 engines
cluster = ipp.Cluster(engines="mpi", n=4)

# Start and connect to the cluster
rc = cluster.start_and_connect_sync()

# Enable IPython magics for parallel processing
rc[:].activate()

Starting 4 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/4 [00:00<?, ?engine/s]

Now we can use the [%%px](https://ipyparallel.readthedocs.io/en/latest/examples/Parallel%20Magics.html#px)
magic on Jupyter cells to interactively run code on the MPI engines.

## In-memory arrays

Let’s create a simple parallel array from a random $4 \times 4$ matrix with $2 \times 2$ chunk size
and store it in local memory:

In [2]:
%%px --block

import numpy as np
import rockverse as rv

sample_array = rv.array(np.random.rand(4, 4), chunks=(2, 2), fill_value=0)

# You can get items just like regular Numpy arrays
sample_array[...]

%px:   0%|          | 0/4 [00:00<?, ?tasks/s]

Out[0:1]: 
array([[0.66519842, 0.04092284, 0.37740439, 0.40443596],
       [0.02200301, 0.60725687, 0.56826518, 0.09376322],
       [0.16654701, 0.76415815, 0.79574127, 0.24875224],
       [0.93306855, 0.59585639, 0.57055363, 0.516995  ]])

Out[1:1]: 
array([[0.66519842, 0.04092284, 0.37740439, 0.40443596],
       [0.02200301, 0.60725687, 0.56826518, 0.09376322],
       [0.16654701, 0.76415815, 0.79574127, 0.24875224],
       [0.93306855, 0.59585639, 0.57055363, 0.516995  ]])

Out[2:1]: 
array([[0.66519842, 0.04092284, 0.37740439, 0.40443596],
       [0.02200301, 0.60725687, 0.56826518, 0.09376322],
       [0.16654701, 0.76415815, 0.79574127, 0.24875224],
       [0.93306855, 0.59585639, 0.57055363, 0.516995  ]])

Out[3:1]: 
array([[0.66519842, 0.04092284, 0.37740439, 0.40443596],
       [0.02200301, 0.60725687, 0.56826518, 0.09376322],
       [0.16654701, 0.76415815, 0.79574127, 0.24875224],
       [0.93306855, 0.59585639, 0.57055363, 0.516995  ]])

Notice that all 4 MPI processes display the same values for sample_array. However, under the hood, RockVerse does not replicate the entire array on each MPI process. Instead, it distributes the array’s chunks across the available processes to optimize memory usage and parallel performance. The `zarray` attribute of the ParallelArray provides direct access to the local Zarr array responsible for managing the data on each individual process:

In [3]:
%%px --block

# This will retrieve the local Zarr array specific to each MPI process
sample_array.zarray[...]

Out[0:2]: 
array([[0.66519842, 0.04092284, 0.        , 0.        ],
       [0.02200301, 0.60725687, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ]])

Out[1:2]: 
array([[0.        , 0.        , 0.37740439, 0.40443596],
       [0.        , 0.        , 0.56826518, 0.09376322],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ]])

Out[3:2]: 
array([[0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.79574127, 0.24875224],
       [0.        , 0.        , 0.57055363, 0.516995  ]])

Out[2:2]: 
array([[0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.16654701, 0.76415815, 0.        , 0.        ],
       [0.93306855, 0.59585639, 0.        , 0.        ]])

In this case, we have 4 chunks and 4 MPI processes, so each process handles one chunk. The zeros shown above correspond to uninitialized chunks, which Zarr represents with every element set to the specified `fill_value`.

We can also inspect information about each local array. Pay close attention to the `Chunks Initialized` detail on each process:

In [4]:
%%px --block

sample_array.zarray.info_complete()

Out[0:3]: 
Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (4, 4)
Chunk shape        : (2, 2)
Order              : C
Read-only          : False
Store type         : MemoryStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 128
No. bytes stored   : 722
Storage ratio      : 0.2
Chunks Initialized : 1

Out[1:3]: 
Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (4, 4)
Chunk shape        : (2, 2)
Order              : C
Read-only          : False
Store type         : MemoryStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 128
No. bytes stored   : 676
Storage ratio      : 0.2
Chunks Initialized : 1

Out[2:3]: 
Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (4, 4)
Chunk shape        : (2, 2)
Order              : C
Read-only          : False
Store type         : MemoryStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 128
No. bytes stored   : 676
Storage ratio      : 0.2
Chunks Initialized : 1

Out[3:3]: 
Type               : Array
Zarr format        : 3
Data type          : Float64(endianness='little')
Fill value         : 0.0
Shape              : (4, 4)
Chunk shape        : (2, 2)
Order              : C
Read-only          : False
Store type         : MemoryStore
Filters            : ()
Serializer         : BytesCodec(endian=<Endian.little: 'little'>)
Compressors        : (ZstdCodec(level=0, checksum=False),)
No. bytes          : 128
No. bytes stored   : 676
Storage ratio      : 0.2
Chunks Initialized : 1

Ideally, you will use ParallelArrays just like regular Numpy arrays, while RockVerse manages all the MPI communications transparently for you.
You can explore the currently implemented operations in the [API documentation](../../../api/core/paralellarray.rst).
For example, let’s set the first column to another number:

In [5]:
%%px --block
sample_array[:, 0] = 3

You can verify that the operation was applied on every process...

In [6]:
%%px --block
sample_array[...]

Out[0:5]: 
array([[3.        , 0.04092284, 0.37740439, 0.40443596],
       [3.        , 0.60725687, 0.56826518, 0.09376322],
       [3.        , 0.76415815, 0.79574127, 0.24875224],
       [3.        , 0.59585639, 0.57055363, 0.516995  ]])

Out[3:5]: 
array([[3.        , 0.04092284, 0.37740439, 0.40443596],
       [3.        , 0.60725687, 0.56826518, 0.09376322],
       [3.        , 0.76415815, 0.79574127, 0.24875224],
       [3.        , 0.59585639, 0.57055363, 0.516995  ]])

Out[2:5]: 
array([[3.        , 0.04092284, 0.37740439, 0.40443596],
       [3.        , 0.60725687, 0.56826518, 0.09376322],
       [3.        , 0.76415815, 0.79574127, 0.24875224],
       [3.        , 0.59585639, 0.57055363, 0.516995  ]])

Out[1:5]: 
array([[3.        , 0.04092284, 0.37740439, 0.40443596],
       [3.        , 0.60725687, 0.56826518, 0.09376322],
       [3.        , 0.76415815, 0.79574127, 0.24875224],
       [3.        , 0.59585639, 0.57055363, 0.516995  ]])

... while each process handles only its own chunk:

In [7]:
%%px --block
sample_array.zarray[...]

Out[0:6]: 
array([[3.        , 0.04092284, 0.        , 0.        ],
       [3.        , 0.60725687, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ]])

Out[2:6]: 
array([[0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [3.        , 0.76415815, 0.        , 0.        ],
       [3.        , 0.59585639, 0.        , 0.        ]])

Out[1:6]: 
array([[0.        , 0.        , 0.37740439, 0.40443596],
       [0.        , 0.        , 0.56826518, 0.09376322],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ]])

Out[3:6]: 
array([[0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.79574127, 0.24875224],
       [0.        , 0.        , 0.57055363, 0.516995  ]])

Array-related properties are also implemented as aliases to the corresponding Zarr array variables
(again, see the [API documentation](../../../api/core/paralellarray.rst) for a comprehensive list):

In [8]:
%%px --block

print(f"""
Data type: {sample_array.dtype}
Shape: {sample_array.shape}
Chunk shape: {sample_array.chunks}
Number of chunks: {sample_array.nchunks}
""")

[stdout:0] 
Data type: float64
Shape: (4, 4)
Chunk shape: (2, 2)
Number of chunks: 4



[stdout:1] 
Data type: float64
Shape: (4, 4)
Chunk shape: (2, 2)
Number of chunks: 4



[stdout:2] 
Data type: float64
Shape: (4, 4)
Chunk shape: (2, 2)
Number of chunks: 4



[stdout:3] 
Data type: float64
Shape: (4, 4)
Chunk shape: (2, 2)
Number of chunks: 4



Keep in mind that calls to RockVerse data are often MPI collective operations, so they must be executed on every process.
Commands like the following:
   
```python
   # THIS IS WRONG!!!
   if mpi_rank == 0:        
       print(sample_array[:, 0])    
```
will cause the execution to stall, because other MPI processes are waiting for their participation (in this case, the `__getitem__` operation).

## Persistent arrays

If the array is stored on a file system, each MPI process will have full access to every chunk via the file system:

In [9]:
%%px --block
sample_array = rv.array(np.random.rand(4, 4),
                        chunks=(2, 2),
                        fill_value=0,
                        store='/path/to/file.zarr')
sample_array.zarray[...]

Out[3:8]: 
array([[0.27980973, 0.92425147, 0.56940082, 0.17134337],
       [0.55884784, 0.28089545, 0.37335043, 0.24030874],
       [0.27919459, 0.9078381 , 0.18492786, 0.46198837],
       [0.46608887, 0.03209672, 0.61565886, 0.20668113]])

Out[0:8]: 
array([[0.27980973, 0.92425147, 0.56940082, 0.17134337],
       [0.55884784, 0.28089545, 0.37335043, 0.24030874],
       [0.27919459, 0.9078381 , 0.18492786, 0.46198837],
       [0.46608887, 0.03209672, 0.61565886, 0.20668113]])

Out[1:8]: 
array([[0.27980973, 0.92425147, 0.56940082, 0.17134337],
       [0.55884784, 0.28089545, 0.37335043, 0.24030874],
       [0.27919459, 0.9078381 , 0.18492786, 0.46198837],
       [0.46608887, 0.03209672, 0.61565886, 0.20668113]])

Out[2:8]: 
array([[0.27980973, 0.92425147, 0.56940082, 0.17134337],
       [0.55884784, 0.28089545, 0.37335043, 0.24030874],
       [0.27919459, 0.9078381 , 0.18492786, 0.46198837],
       [0.46608887, 0.03209672, 0.61565886, 0.20668113]])

Processing strategy, however, remains the same: each chunk is handled by a single MPI process, and this distribution is transparent to the user.

## Working with attributes

RockVerse provides a wrapper to manage attributes in parallel arrays using a dict-like interface. You can access it through the `attrs` property:

In [10]:
%%px --block
sample_array.attrs

Out[1:9]: Attributes({'_ROCKVERSE_DATATYPE': 'ParallelArray'})

Out[3:9]: Attributes({'_ROCKVERSE_DATATYPE': 'ParallelArray'})

Out[2:9]: Attributes({'_ROCKVERSE_DATATYPE': 'ParallelArray'})

Out[0:9]: Attributes({'_ROCKVERSE_DATATYPE': 'ParallelArray'})

You can assign custom key/value attributes to the `attrs` property, provided that the values are JSON serializable:

In [11]:
%%px --block
sample_array.attrs['foo'] = 'bar'
print(sample_array.attrs['foo'])

[stdout:2] bar


[stdout:3] bar


[stdout:0] bar


[stdout:1] bar


Under the hood, attributes are stored only by MPI rank 0, while the other ranks participate in collective MPI calls. 
This design ensures synchronized access and modification of attributes across MPI processes, maintaining consistency and 
preventing race conditions when working with data on the local file system:

In [12]:
%%px --block

# This will print the attributes from the
# local Zarr array on each MPI process
print(sample_array.zarray.attrs.asdict())

[stdout:0] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar'}


[stdout:1] {}


[stdout:2] {}


[stdout:3] {}


In [13]:
%%px --block

# This will print the attributes from the parallel array,
# which will be the same across all the MPI processes
print(sample_array.attrs.asdict())

[stdout:3] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar'}


[stdout:0] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar'}


[stdout:2] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar'}


[stdout:1] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar'}


Just like with the array values, you only need to interact with the parallel array’s `attrs` property and let RockVerse handle MPI synchronization for you.

Calls to RockVerse attributes are collective operations, so they must be executed on every process.
Commands like the following:

```python    
# THIS IS WRONG!!!
if mpi_rank == 0:        
     print(sample_array.attrs.asdict())    
```
   
will cause the execution to stall, as other MPI processes wait to participate in retrieving the attributes.

## Special metadata


There are five special attribute keys in parallel arrays that RockVerse 
uses for plotting and exporting to other formats:

- `name`: the array name;
- `unit`: the data units for the array values;
- `latex_name`: a LaTeX string representation of name;
- `latex_unit`: a LaTeX string representation of unit;
- `description`: a free-text string describing the data.

These attributes are optional and can be accessed or modified using the corresponding properties:

In [14]:
%%px --block

sample_array.name = 'rho'
sample_array.unit = 'kg/m3'
sample_array.latex_name = r'$\rho$'
sample_array.latex_unit = 'kg/m$^3$'
sample_array.description = 'Fluid density'
print(sample_array.attrs.asdict())

[stdout:1] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar', 'name': 'rho', 'unit': 'kg/m3', 'latex_name': '$\\rho$', 'latex_unit': 'kg/m$^3$', 'description': 'Fluid density'}


[stdout:0] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar', 'name': 'rho', 'unit': 'kg/m3', 'latex_name': '$\\rho$', 'latex_unit': 'kg/m$^3$', 'description': 'Fluid density'}


[stdout:2] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar', 'name': 'rho', 'unit': 'kg/m3', 'latex_name': '$\\rho$', 'latex_unit': 'kg/m$^3$', 'description': 'Fluid density'}


[stdout:3] {'_ROCKVERSE_DATATYPE': 'ParallelArray', 'foo': 'bar', 'name': 'rho', 'unit': 'kg/m3', 'latex_name': '$\\rho$', 'latex_unit': 'kg/m$^3$', 'description': 'Fluid density'}


Those values can be quickly retrieved, for example, when building data plots:

In [15]:
%%px --block
sample_array.get_plot_label()

Out[2:14]: '$\\rho$ (kg/m$^3$)'

Out[3:14]: '$\\rho$ (kg/m$^3$)'

Out[0:14]: '$\\rho$ (kg/m$^3$)'

Out[1:14]: '$\\rho$ (kg/m$^3$)'

or when exporting data to a LAS file in case of well logs.

## Exporting to other formats

Exporting parallel arrays to other formats is straightforward with RockVerse. The library provides built-in methods that 
handle the complexities of parallel I/O and synchronization across MPI processes, allowing you to save your data seamlessly 
into standard file formats without needing to manage low-level details. This simplifies integration with other tools and 
workflows that rely on common scientific data formats. The following options are currently available, and more formats 
are comming in future releases.

### HDF5

The [h5_dump](../../../api/core/_autogen/rockverse.ParallelArray.h5_dump.rst) method will write the 
HDF5 dataset with the chunked array data and all the attributes using the 
[h5py](https://docs.h5py.org/en/stable/index.html) library:

In [16]:
%%px --block

sample_array.h5_dump(filename='/path/to/filename.h5',
                     path='/internal/hdf5/path/to/dataset',
                     mode='w') #<- overwrite mode

Let's load this data from the HDF5 file and check what is inside:

In [20]:
# Note that here we are not using the %%px magic, so this part
# will run in the local namespace (outside the cluster)

import h5py

with h5py.File('/path/to/filename.h5', mode='r') as fobj:
    array = fobj['/internal/hdf5/path/to/dataset']

    print(f"""Array info in the HDF5 file:
    Type: {array.dtype}
    Shape: {array.shape}
    Chunk shape: {array.chunks}
    Values:\n{array[...]}
    """)
    print('Attributes:\n',
            '\n'.join([f"""    {k}: {v}""" for k, v in array.attrs.items()]))


Array info in the HDF5 file:
    Type: float64
    Shape: (4, 4)
    Chunk shape: (2, 2)
    Values:
[[0.27980973 0.92425147 0.56940082 0.17134337]
 [0.55884784 0.28089545 0.37335043 0.24030874]
 [0.27919459 0.9078381  0.18492786 0.46198837]
 [0.46608887 0.03209672 0.61565886 0.20668113]]
    
Attributes:
     _ROCKVERSE_DATATYPE: ParallelArray
    description: Fluid density
    foo: bar
    latex_name: $\rho$
    latex_unit: kg/m$^3$
    name: rho
    unit: kg/m3
